In [ ]:
import re
import joblib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from urllib.parse import urlparse
from difflib import SequenceMatcher

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

In [ ]:
WEB_DATA_PATH = "PhiUSIIL_Phishing_URL_Dataset.csv"

df_web = pd.read_csv(WEB_DATA_PATH)

print("Web dataset shape:", df_web.shape)
display(df_web.head())

In [ ]:
print("Dataset info:\n")
df_web.info()

print("\nDuplicate rows:", df_web.duplicated().sum())
print("Duplicate URLs:", df_web["URL"].duplicated().sum())
print("Duplicate Domains:", df_web["Domain"].duplicated().sum())
print("Duplicate Titles:", df_web["Title"].duplicated().sum())

print("\nTotal null values:", df_web.isnull().sum().sum())

print("\nClass counts:")
print(df_web["label"].value_counts())

print("\nClass percentages:")
print((df_web["label"].value_counts(normalize=True) * 100).round(2))

In [ ]:
web_drop_cols = [
    "FILENAME",
    "URL",
    "Domain",
    "Title",
    "TLD",
    "URLSimilarityIndex",
    "TLDLegitimateProb",
    "URLCharProb",
    "DomainTitleMatchScore",
    "URLTitleMatchScore",
    "LineOfCode",
    "LargestLineLength",
    "HasTitle",
    "HasCopyrightInfo",
    "NoOfImage",
    "NoOfCSS",
    "NoOfJS",
    "NoOfSelfRef",
    "NoOfEmptyRef",
    "NoOfExternalRef"
]

df_web_model = df_web.drop(columns=web_drop_cols, errors="ignore")

print("Shape after dropping columns:", df_web_model.shape)
print("Remaining columns:")
print(df_web_model.columns.tolist())

In [ ]:
X_web_all = df_web_model.drop(columns=["label"])
y_web_all = df_web_model["label"]

print("Feature matrix shape:", X_web_all.shape)
print("Target shape:", y_web_all.shape)

In [ ]:
groups = df_web["Domain"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in gss.split(X_web_all, y_web_all, groups=groups):
    X_train_web = X_web_all.iloc[train_idx].copy()
    X_test_web = X_web_all.iloc[test_idx].copy()
    y_train_web = y_web_all.iloc[train_idx].copy()
    y_test_web = y_web_all.iloc[test_idx].copy()
    train_domains = groups.iloc[train_idx]
    test_domains = groups.iloc[test_idx]

print("Web train shape:", X_train_web.shape)
print("Web test shape:", X_test_web.shape)

train_domain_set = set(train_domains.unique())
test_domain_set = set(test_domains.unique())
domain_overlap = train_domain_set.intersection(test_domain_set)

print("Unique train domains:", len(train_domain_set))
print("Unique test domains:", len(test_domain_set))
print("Overlapping domains:", len(domain_overlap))

In [ ]:
web_models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.5,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=1,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=1,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
web_results = []
trained_web_models = {}

for name, model in web_models.items():
    print(f"\n{'=' * 60}")
    print(f"Training {name}...")

    model.fit(X_train_web, y_train_web)
    trained_web_models[name] = model

    y_pred = model.predict(X_test_web)
    y_prob = model.predict_proba(X_test_web)[:, 1]

    acc = accuracy_score(y_test_web, y_pred)
    precision = precision_score(y_test_web, y_pred, zero_division=0)
    recall = recall_score(y_test_web, y_pred, zero_division=0)
    f1 = f1_score(y_test_web, y_pred, zero_division=0)
    roc = roc_auc_score(y_test_web, y_prob)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_web, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test_web, y_pred, zero_division=0))

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc:.4f}")

    web_results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc
    })

In [ ]:
web_results_df = pd.DataFrame(web_results)
web_results_df = web_results_df.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)

display(web_results_df.round(4))

best_web_model_name = web_results_df.iloc[0]["Model"]
best_web_model = trained_web_models[best_web_model_name]

print("Best web model:", best_web_model_name)

In [ ]:
web_columns = X_train_web.columns.tolist()

joblib.dump(best_web_model, "best_web_model.pkl")
joblib.dump(web_columns, "web_model_columns.pkl")

print("Web model and columns saved")
print("Web feature count:", len(web_columns))

In [ ]:
URL_TRAIN_PATH = "train_dataset.csv"
URL_TEST_PATH = "test_dataset.csv"

df_train = pd.read_csv(URL_TRAIN_PATH)
df_test = pd.read_csv(URL_TEST_PATH)

print("URL train shape:", df_train.shape)
print("URL test shape:", df_test.shape)

display(df_train.head())
display(df_test.head())

In [ ]:
print("Train duplicate rows:", df_train.duplicated().sum())
print("Test duplicate rows:", df_test.duplicated().sum())

print("\nTrain total null values:", df_train.isnull().sum().sum())
print("Test total null values:", df_test.isnull().sum().sum())

print("\nTrain label counts:")
print(df_train["label"].value_counts())

print("\nTrain label percentages:")
print((df_train["label"].value_counts(normalize=True) * 100).round(2))

print("\nTest label counts:")
print(df_test["label"].value_counts())

print("\nTest label percentages:")
print((df_test["label"].value_counts(normalize=True) * 100).round(2))

In [ ]:
url_drop_cols = [
    "url",
    "source",
    "tld",
    "url_3bentropy",
    "url_hamming_01",
    "url_hamming_10",
    "url_hamming_11"
]

In [ ]:
X_train_url = df_train.drop(columns=url_drop_cols + ["label"], errors="ignore")
y_train_url = df_train["label"]

X_test_url = df_test.drop(columns=url_drop_cols + ["label"], errors="ignore")
y_test_url = df_test["label"]

print("URL train feature shape:", X_train_url.shape)
print("URL test feature shape:", X_test_url.shape)

In [ ]:
for col in X_train_url.columns:
    if col not in X_test_url.columns:
        X_test_url[col] = 0

extra_test_cols = [c for c in X_test_url.columns if c not in X_train_url.columns]
if extra_test_cols:
    X_test_url = X_test_url.drop(columns=extra_test_cols)

X_test_url = X_test_url[X_train_url.columns]

print("Aligned URL train shape:", X_train_url.shape)
print("Aligned URL test shape:", X_test_url.shape)
print("Columns match:", list(X_train_url.columns) == list(X_test_url.columns))

In [ ]:
int_cols_train = X_train_url.select_dtypes(include=["int64"]).columns
float_cols_train = X_train_url.select_dtypes(include=["float64"]).columns

X_train_url[int_cols_train] = X_train_url[int_cols_train].astype(np.int32)
X_test_url[int_cols_train] = X_test_url[int_cols_train].astype(np.int32)

X_train_url[float_cols_train] = X_train_url[float_cols_train].astype(np.float32)
X_test_url[float_cols_train] = X_test_url[float_cols_train].astype(np.float32)

print(X_train_url.dtypes.value_counts())

In [ ]:
neg = (y_train_url == 0).sum()
pos = (y_train_url == 1).sum()
scale_pos_weight = neg / pos

print("Negative samples:", neg)
print("Positive samples:", pos)
print("scale_pos_weight:", round(scale_pos_weight, 4))

In [ ]:
url_models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.5,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=700,
        max_depth=7,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=5,
        gamma=0.5,
        reg_alpha=0.05,
        reg_lambda=1.5,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
url_results = []
trained_url_models = {}

for name, model in url_models.items():
    print(f"\n{'=' * 60}")
    print(f"Training {name} on URL training dataset...")

    model.fit(X_train_url, y_train_url)
    trained_url_models[name] = model

    y_pred = model.predict(X_test_url)
    y_prob = model.predict_proba(X_test_url)[:, 1]

    acc = accuracy_score(y_test_url, y_pred)
    precision = precision_score(y_test_url, y_pred, zero_division=0)
    recall = recall_score(y_test_url, y_pred, zero_division=0)
    f1 = f1_score(y_test_url, y_pred, zero_division=0)
    roc = roc_auc_score(y_test_url, y_prob)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_url, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test_url, y_pred, zero_division=0))

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc:.4f}")

    url_results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc
    })

In [ ]:
url_results_df = pd.DataFrame(url_results)
url_results_df = url_results_df.sort_values(by="Recall", ascending=False).reset_index(drop=True)

display(url_results_df.round(4))

best_url_model_name = url_results_df.iloc[0]["Model"]
best_url_model = trained_url_models[best_url_model_name]

print("Best URL model:", best_url_model_name)